# Lab 3 - Exact NN vs. HNSW
## Step 1 - Install Library

In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2683497 sha256=bf097c8eefc88cf07f8a0627a654e9495cdde54ef8779554d39f8c3ac1f4a395
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


## Step 2 - Import Libraries



In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

## Step 3 - Create Dummy Dataset

In [3]:
# Build Dummy Dataset
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # search for 5 nearest neighbors

## Step 4 - Construct NN Model

In [4]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performance
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Time:", end - start, "s")

=== Exact NN ===
Indices: [[268  85 845 776 342]]
Distances: [[0.01719012 0.02547052 0.02814774 0.02893231 0.03845241]]
Time: 0.03753185272216797 s


## Step 5 - Construct HNSW Model

In [5]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Time:", end - start, "s")


=== HNSW ===
Indices: [[268  85 845 776 342]]
Distances: [[0.0002955  0.00064875 0.0007923  0.00083708 0.00147859]]
Time: 0.00015854835510253906 s
